[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/C.%20Transportation%2C%20Routing%20%26%20Freight%20Management%20%28The%20Physical%20Internet%29/081.%20The%203D%20Container_Truck%20Loading%20Problem%20%28Bin%20Packing%29/P81-Tier-2_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---

# 81. The 3D Container/Truck Loading Problem

## Tier 2: Classic Heuristic (Bottom-Left-Fill Algorithm)

This notebook builds a feasible 3D loading plan using a Bottom-Left-Fill (BLF) heuristic. Items are considered in priority order. For each item, the algorithm tests a finite set of candidate positions and axis-aligned orientations, rejects placements that violate the modeled constraints, and chooses the feasible candidate with the lowest position score.

**Inputs:** container dimensions and weight capacity; item dimensions, weights, identifiers, and urgency factors; and the BLF parameter `alpha` controlling randomized priority-order diversification.

**Outputs:** the items placed, their positions and orientations, a placement trace, utilization and center-of-gravity metrics, and visualizations of the resulting arrangement.

**Scope and modeling assumptions:** Items are rectangular cuboids, may be rotated by permuting their three dimensions, and remain axis-aligned. The container is a rectangular box. Items cannot overlap or exceed its boundaries, and the total packed weight cannot exceed the container capacity. A placement above the floor must have at least 50% of its bottom-face area directly supported by items beneath it. This area rule is a simplified stability proxy; it does not model load-bearing limits, friction, tipping moments, axle loads, or access and unloading constraints. The heuristic can leave items unpacked and does not generally certify a globally optimal arrangement.

### Key ideas

- **Priority ordering:** the score is volume × density × urgency factor. Since volume × density equals weight, this score is numerically weight × urgency factor. Larger scores are considered first.
- **Support points:** candidate coordinates are generated from the corners and selected edge midpoints of placed items, starting at the container origin. Every candidate is checked against the actual bounds, overlap, weight, and support rules.
- **Position score:** feasible positions are ranked using normalized height, distance from the origin in the horizontal plane, and 3D distance from the origin. A lower score is preferred; it is a heuristic position score, not a direct measurement of empty volume.
- **Controlled randomness:** `alpha` sets the probability of entering the adjacent-swap diversification step and the probability of each adjacent swap. Randomness can change item order and therefore the resulting plan.

### Concrete example

The container measures 10 × 8 × 6 units and has a 1,000 kg weight capacity. The three items are:

| Item | Dimensions (units) | Weight (kg) | Urgency factor | Priority score |
|---|---:|---:|---:|---:|
| A | 4 × 3 × 2 | 10 | 1.0 | 10 |
| B | 3 × 3 × 3 | 15 | 1.0 | 15 |
| C | 2 × 4 × 3 | 8 | 1.0 | 8 |

For example, Item A has volume 24 cubic units and density $10/24$ kg per cubic unit, so its priority is $24(10/24)(1.0)=10$. The priority is a sequencing rule, not a measure of packing quality.

## The BLF decision rule

A *position* is the lower-left-bottom coordinate $(x,y,z)$ of an item. An *orientation* $(l,w,h)$ assigns the item's three side lengths to the container's length, width, and height axes. The algorithm does not search every possible coordinate: it tests positions in a dynamically generated support-point set $P$ and the distinct axis-aligned orientations $O_i$ of the current item $i$.

For a candidate placement, the position score is

$$
S(x,y,z) = 0.4\frac{z}{H}
+ 0.4\frac{x+y}{L+W}
+ 0.2\frac{\sqrt{x^2+y^2+z^2}}{\sqrt{L^2+W^2+H^2}}.
$$

Here $L,W,H$ are the container dimensions. The three terms favor lower placements, positions nearer the horizontal origin, and positions nearer the 3D origin, respectively. The denominators normalize the terms by container dimensions; the weights sum to one. Item dimensions do not appear in this score: they affect whether a candidate is feasible, while the score ranks feasible candidate origins.

Let $P_t$ be the candidate support points available when item $i_t$ is considered, and let $F(i_t,p,o)$ mean that placing item $i_t$ at point $p$ in orientation $o$ satisfies all modeled feasibility checks. The selection rule is

$$
(p_t^*,o_t^*) \in
\underset{p\in P_t,\,o\in O_{i_t}:\,F(i_t,p,o)}{\operatorname{argmin}}
S(p).
$$

If no feasible candidate exists, the item is left unpacked. If several candidates have the same score, the implementation keeps the first one encountered. The finite support-point search is what makes the procedure a heuristic rather than a proof of optimality over all continuous positions.

### Feasibility checks

For an item with lower corner $(x,y,z)$ and oriented dimensions $(l,w,h)$, the boundary conditions are

$$
0\le x,\quad 0\le y,\quad 0\le z,\qquad
x+l\le L,\quad y+w\le W,\quad z+h\le H.
$$

Two axis-aligned boxes do not overlap if they are separated along at least one axis. Thus, for boxes 1 and 2, overlap occurs exactly when their intervals overlap in all three dimensions. For an item above the floor, let $A_i=lw$ be its bottom-face area and let $A_i^{\mathrm{sup}}$ be the summed area of intersections between that face and the top faces of items directly below it. The notebook's support rule is

$$
A_i^{\mathrm{sup}}/A_i \ge 0.5.
$$

An item at $z=0$ is supported by the container floor. Finally, for packed set $Q$, the weight constraint is $\sum_{j\in Q} w_j+w_i\le C$, where $w_j$ is item weight and $C$ is the container's maximum weight.

### Algorithm flow and computational effort

The algorithm sorts items by priority, then processes them one at a time. It evaluates the Cartesian product of the current support points and the item's distinct orientations, chooses the lowest-score feasible candidate, places the item if one exists, and generates additional support points. When the support-point list exceeds the pruning threshold, the implementation retains the 100 points ranked first by height and then by $x+y$. This is a bounded shortlist heuristic, not a mathematical test that every discarded point is dominated.

If there are $n$ items, at most six distinct orientations per item, at most $K$ support points, and up to $n$ already placed items to check, the placement checks take at most $O(n^2K)$ time over the full run, in addition to sorting and support-point management. In this implementation, pruning limits the retained list to at most 100 points at pruning time; generating points after a placement can temporarily increase it. This bound describes the implemented candidate search, not the complexity of the general 3D packing problem. The heuristic can miss feasible placements because it searches only its candidate points and can discard points during pruning.

In [ ]:
import time
from itertools import permutations

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

# A fixed seed makes the randomized priority-order step reproducible.
np.random.seed(42)

## Representing items, the container, and candidate points

The classes below keep the geometric and operational data together. Dimensions use the same length unit throughout, so item and container volumes use cubic units. Weights use kilograms. `get_orientations` generates distinct permutations of an item's side lengths; repeated dimensions therefore do not create duplicate orientations.

In [ ]:
class Item:
    """A rectangular item that can be axis-aligned in any distinct orientation."""

    def __init__(self, length, width, height, weight, item_id, urgency_factor=1.0):
        if min(length, width, height) <= 0:
            raise ValueError("Item dimensions must be positive.")
        if weight <= 0:
            raise ValueError("Item weight must be positive.")
        if urgency_factor <= 0:
            raise ValueError("The urgency factor must be positive.")

        self.length = length
        self.width = width
        self.height = height
        self.weight = weight
        self.item_id = item_id
        self.urgency_factor = urgency_factor
        self.position = None
        self.orientation = None

    def volume(self):
        return self.length * self.width * self.height

    def density(self):
        return self.weight / self.volume()

    def get_orientations(self):
        """Return distinct axis-aligned orientations in a reproducible order."""
        dimensions = (self.length, self.width, self.height)
        return list(dict.fromkeys(permutations(dimensions)))

    def calculate_priority(self):
        """Return volume times density times urgency factor."""
        return self.volume() * self.density() * self.urgency_factor

    def max_dimension(self):
        return max(self.length, self.width, self.height)

    def min_dimension(self):
        return min(self.length, self.width, self.height)

    def __repr__(self):
        return (
            f"Item{self.item_id}({self.length}×{self.width}×{self.height}, "
            f"{self.weight}kg, priority={self.calculate_priority():.2f})"
        )


class Container:
    """A rectangular container with an optional maximum weight capacity."""

    def __init__(self, length, width, height, max_weight=None):
        if min(length, width, height) <= 0:
            raise ValueError("Container dimensions must be positive.")
        if max_weight is not None and max_weight <= 0:
            raise ValueError("A finite maximum weight must be positive.")

        self.length = length
        self.width = width
        self.height = height
        self.max_weight = float("inf") if max_weight is None else max_weight

    def volume(self):
        return self.length * self.width * self.height

    def __repr__(self):
        return (
            f"Container({self.length}×{self.width}×{self.height}, "
            f"max_weight={self.max_weight}kg)"
        )


class SupportPoint:
    """A candidate lower-corner coordinate for a new item placement."""

    def __init__(self, x, y, z, source_item=None):
        self.x = x
        self.y = y
        self.z = z
        self.source_item = source_item

    def __repr__(self):
        return f"SupportPoint({self.x}, {self.y}, {self.z})"

    def __eq__(self, other):
        if not isinstance(other, SupportPoint):
            return NotImplemented
        return (
            abs(self.x - other.x) < 1e-6
            and abs(self.y - other.y) < 1e-6
            and abs(self.z - other.z) < 1e-6
        )

    def __hash__(self):
        return hash((round(self.x, 6), round(self.y, 6), round(self.z, 6)))

## Implementing the Bottom-Left-Fill heuristic

`BottomLeftFill3D` keeps the current packed set and support-point list. Its feasibility checks implement the model constraints before a candidate can be scored. The sequential decision rule matters: each placement changes the geometry and candidate points available to later items, so the order can affect the final plan.

In [ ]:
class BottomLeftFill3D:
    """Bottom-Left-Fill heuristic for axis-aligned 3D container loading."""

    def __init__(self, container, alpha=0.25):
        if not 0 <= alpha <= 1:
            raise ValueError("alpha must be between 0 and 1.")

        self.container = container
        self.alpha = alpha
        self.packed_items = []
        self.support_points = [SupportPoint(0, 0, 0)]
        self.placement_history = []

    def sort_by_priority(self, items):
        """Sort by descending priority, with controlled adjacent-swap diversification."""
        item_priorities = [
            (item, item.calculate_priority()) for item in items
        ]
        sorted_items = sorted(
            item_priorities,
            key=lambda item_priority: item_priority[1],
            reverse=True,
        )

        if np.random.random() < self.alpha:
            for index in range(len(sorted_items) - 1):
                if np.random.random() < self.alpha:
                    sorted_items[index], sorted_items[index + 1] = (
                        sorted_items[index + 1],
                        sorted_items[index],
                    )

        return [item for item, _ in sorted_items]

    def is_feasible_position(self, item, position, orientation):
        """Check weight, container boundaries, overlap, and the support rule."""
        x, y, z = position
        length, width, height = orientation

        packed_weight = sum(packed_item.weight for packed_item in self.packed_items)
        if packed_weight + item.weight > self.container.max_weight:
            return False

        if (
            x < 0
            or y < 0
            or z < 0
            or x + length > self.container.length
            or y + width > self.container.width
            or z + height > self.container.height
        ):
            return False

        for existing_item in self.packed_items:
            if self.items_overlap(item, position, orientation, existing_item):
                return False

        if z > 0 and not self.has_adequate_support(item, position, orientation):
            return False

        return True

    def items_overlap(self, item1, pos1, orient1, item2):
        """Return whether two axis-aligned rectangular boxes intersect in volume."""
        if item1 is item2:
            return False

        x1, y1, z1 = pos1
        length1, width1, height1 = orient1

        x2, y2, z2 = item2.position
        length2, width2, height2 = item2.orientation

        overlap_x = not (x1 + length1 <= x2 or x2 + length2 <= x1)
        overlap_y = not (y1 + width1 <= y2 or y2 + width2 <= y1)
        overlap_z = not (z1 + height1 <= z2 or z2 + height2 <= z1)

        return overlap_x and overlap_y and overlap_z

    def has_adequate_support(self, item, position, orientation):
        """Apply the model's rule requiring at least 50% bottom-face support."""
        x, y, z = position
        length, width, _ = orientation

        if z == 0:
            return True

        support_area = 0
        item_bottom_area = length * width

        for existing_item in self.packed_items:
            ex_x, ex_y, ex_z = existing_item.position
            ex_length, ex_width, ex_height = existing_item.orientation

            if abs(z - (ex_z + ex_height)) < 0.001:
                overlap_x = max(
                    0,
                    min(x + length, ex_x + ex_length) - max(x, ex_x),
                )
                overlap_y = max(
                    0,
                    min(y + width, ex_y + ex_width) - max(y, ex_y),
                )
                support_area += overlap_x * overlap_y

        return support_area / item_bottom_area >= 0.5

    def calculate_wasted_space(self, item, position, orientation):
        """Return the normalized position score used to rank feasible placements."""
        x, y, z = position

        gravity_score = z / self.container.height
        bottom_left_score = (x + y) / (
            self.container.length + self.container.width
        )
        corner_distance = np.sqrt(x**2 + y**2 + z**2)
        maximum_distance = np.sqrt(
            self.container.length**2
            + self.container.width**2
            + self.container.height**2
        )
        compactness_score = corner_distance / maximum_distance

        return (
            0.4 * gravity_score
            + 0.4 * bottom_left_score
            + 0.2 * compactness_score
        )

    def update_support_points(self, placed_item):
        """Generate candidate coordinates from the placed item's geometry."""
        x, y, z = placed_item.position
        length, width, height = placed_item.orientation

        new_points = [
            SupportPoint(x + length, y, z, placed_item),
            SupportPoint(x, y + width, z, placed_item),
            SupportPoint(x, y, z + height, placed_item),
            SupportPoint(x + length, y + width, z, placed_item),
            SupportPoint(x + length, y, z + height, placed_item),
            SupportPoint(x, y + width, z + height, placed_item),
            SupportPoint(x + length, y + width, z + height, placed_item),
            SupportPoint(x + length / 2, y, z, placed_item),
            SupportPoint(x, y + width / 2, z, placed_item),
            SupportPoint(x + length, y + width / 2, z, placed_item),
            SupportPoint(x + length / 2, y + width, z, placed_item),
        ]

        for point in new_points:
            if point not in self.support_points:
                self.support_points.append(point)

        self.remove_dominated_points()

    def remove_dominated_points(self):
        """Prune the candidate list by height and horizontal distance ranking."""
        if len(self.support_points) <= 50:
            return

        sorted_points = sorted(
            self.support_points,
            key=lambda point: (point.z, point.x + point.y),
        )
        self.support_points = sorted_points[:100]

    def pack_items(self, items):
        """Pack items sequentially using priority order and support-point search."""
        print(f"Starting BLF 3D packing for {len(items)} items...")
        start_time = time.time()

        sorted_items = self.sort_by_priority(items)
        priority_order = [
            f"Item{item.item_id}({item.calculate_priority():.2f})"
            for item in sorted_items
        ]
        print(f"Item priority order: {priority_order}")

        for item in sorted_items:
            print(
                f"\nPacking Item {item.item_id} "
                f"(Priority: {item.calculate_priority():.2f})"
            )

            best_position = None
            best_orientation = None
            minimum_score = float("inf")
            positions_evaluated = 0

            for support_point in self.support_points:
                position = (support_point.x, support_point.y, support_point.z)

                for orientation in item.get_orientations():
                    positions_evaluated += 1

                    if self.is_feasible_position(item, position, orientation):
                        score = self.calculate_wasted_space(
                            item, position, orientation
                        )
                        if score < minimum_score:
                            minimum_score = score
                            best_position = position
                            best_orientation = orientation

            if best_position is not None:
                item.position = best_position
                item.orientation = best_orientation
                self.packed_items.append(item)

                print(
                    f"  Placed at {best_position} "
                    f"with orientation {best_orientation}"
                )
                print(f"  Position score: {minimum_score:.4f}")

                self.placement_history.append(
                    {
                        "item_id": item.item_id,
                        "position": best_position,
                        "orientation": best_orientation,
                        "priority": item.calculate_priority(),
                        "waste_score": minimum_score,
                        "support_points_count": len(self.support_points),
                    }
                )

                self.update_support_points(item)
                print(
                    "  Support points after update: "
                    f"{len(self.support_points)}"
                )
            else:
                print(f"  Could not place Item {item.item_id}")

            print(f"  Candidate placements evaluated: {positions_evaluated}")

        elapsed_time = time.time() - start_time
        print(f"\nBLF algorithm completed in {elapsed_time:.3f} seconds")
        return self.packed_items

## Measuring a packing plan

Volume utilization is the total volume of packed items divided by container volume. Weight utilization is defined only when the container has a finite capacity. The center of gravity is computed as the weight-weighted average of the centers of the packed items; it describes the load's center of mass under the model's uniform-item-density assumption and is not a vehicle axle-load calculation.

In [ ]:
def calculate_solution_metrics(packed_items, container):
    """Calculate volume, weight, utilization, and load center metrics."""
    if not packed_items:
        return {
            "total_volume": 0,
            "total_weight": 0,
            "volume_utilization": 0,
            "weight_utilization": 0,
            "items_packed": 0,
            "center_of_gravity": (0, 0, 0),
        }

    total_volume = sum(item.volume() for item in packed_items)
    total_weight = sum(item.weight for item in packed_items)
    volume_utilization = total_volume / container.volume() * 100
    weight_utilization = (
        total_weight / container.max_weight * 100
        if np.isfinite(container.max_weight)
        else 0
    )

    weighted_x = 0
    weighted_y = 0
    weighted_z = 0

    for item in packed_items:
        x, y, z = item.position
        length, width, height = item.orientation

        item_center_x = x + length / 2
        item_center_y = y + width / 2
        item_center_z = z + height / 2

        weighted_x += item_center_x * item.weight
        weighted_y += item_center_y * item.weight
        weighted_z += item_center_z * item.weight

    center_of_gravity = (
        weighted_x / total_weight,
        weighted_y / total_weight,
        weighted_z / total_weight,
    )

    return {
        "total_volume": total_volume,
        "total_weight": total_weight,
        "volume_utilization": volume_utilization,
        "weight_utilization": weight_utilization,
        "items_packed": len(packed_items),
        "center_of_gravity": center_of_gravity,
    }

## Visualizing the arrangement

The four views show the same placements from different directions. Each item has a distinct color and a matching legend entry. The 3D view uses the container's actual axis limits and aspect ratio so that the display reflects its dimensions.

In [ ]:
def visualize_blf_solution(
    packed_items,
    container,
    title="3D Container Loading - BLF Solution",
):
    """Draw the packing in 3D and in three orthogonal projections."""
    figure = plt.figure(figsize=(15, 10))
    axis_3d = figure.add_subplot(221, projection="3d")

    container_corners = [
        (0, 0, 0),
        (container.length, 0, 0),
        (container.length, container.width, 0),
        (0, container.width, 0),
        (0, 0, container.height),
        (container.length, 0, container.height),
        (container.length, container.width, container.height),
        (0, container.width, container.height),
    ]
    edges = [
        (0, 1), (1, 2), (2, 3), (3, 0),
        (4, 5), (5, 6), (6, 7), (7, 4),
        (0, 4), (1, 5), (2, 6), (3, 7),
    ]

    for start_index, end_index in edges:
        edge_points = [
            container_corners[start_index],
            container_corners[end_index],
        ]
        axis_3d.plot(
            *zip(*edge_points),
            color="black",
            alpha=0.4,
            linewidth=1.5,
        )

    color_values = plt.cm.rainbow(
        np.linspace(0, 1, max(len(packed_items), 1))
    )
    legend_handles = []

    for index, item in enumerate(packed_items):
        x, y, z = item.position
        length, width, height = item.orientation
        color = color_values[index]

        vertices = [
            (x, y, z),
            (x + length, y, z),
            (x + length, y + width, z),
            (x, y + width, z),
            (x, y, z + height),
            (x + length, y, z + height),
            (x + length, y + width, z + height),
            (x, y + width, z + height),
        ]
        faces = [
            [vertices[0], vertices[1], vertices[5], vertices[4]],
            [vertices[2], vertices[3], vertices[7], vertices[6]],
            [vertices[0], vertices[3], vertices[7], vertices[4]],
            [vertices[1], vertices[2], vertices[6], vertices[5]],
            [vertices[0], vertices[1], vertices[2], vertices[3]],
            [vertices[4], vertices[5], vertices[6], vertices[7]],
        ]
        axis_3d.add_collection3d(
            Poly3DCollection(
                faces,
                facecolors=color,
                edgecolors="black",
                linewidths=0.5,
                alpha=0.75,
            )
        )
        axis_3d.text(
            x + length / 2,
            y + width / 2,
            z + height / 2,
            f"I{item.item_id}",
            fontsize=9,
            ha="center",
            va="center",
        )
        legend_handles.append(
            Patch(facecolor=color, edgecolor="black", label=f"Item {item.item_id}")
        )

    axis_3d.set_xlim(0, container.length)
    axis_3d.set_ylim(0, container.width)
    axis_3d.set_zlim(0, container.height)
    axis_3d.set_box_aspect(
        (container.length, container.width, container.height)
    )
    axis_3d.set_xlabel("Length (units)")
    axis_3d.set_ylabel("Width (units)")
    axis_3d.set_zlabel("Height (units)")
    axis_3d.set_title(f"{title}\n3D view")
    if legend_handles:
        axis_3d.legend(handles=legend_handles, loc="upper left", fontsize=8)

    top_axis = figure.add_subplot(222)
    side_axis = figure.add_subplot(223)
    front_axis = figure.add_subplot(224)

    for index, item in enumerate(packed_items):
        x, y, z = item.position
        length, width, height = item.orientation
        color = color_values[index]

        top_axis.add_patch(
            plt.Rectangle(
                (x, y), length, width,
                facecolor=color, alpha=0.75, edgecolor="black",
            )
        )
        top_axis.text(x + length / 2, y + width / 2, f"I{item.item_id}",
                      ha="center", va="center")

        side_axis.add_patch(
            plt.Rectangle(
                (x, z), length, height,
                facecolor=color, alpha=0.75, edgecolor="black",
            )
        )
        side_axis.text(x + length / 2, z + height / 2, f"I{item.item_id}",
                       ha="center", va="center")

        front_axis.add_patch(
            plt.Rectangle(
                (y, z), width, height,
                facecolor=color, alpha=0.75, edgecolor="black",
            )
        )
        front_axis.text(y + width / 2, z + height / 2, f"I{item.item_id}",
                        ha="center", va="center")

    top_axis.set_xlim(0, container.length)
    top_axis.set_ylim(0, container.width)
    top_axis.set_xlabel("Length (units)")
    top_axis.set_ylabel("Width (units)")
    top_axis.set_title("Top view (XY plane)")
    top_axis.set_aspect("equal")
    top_axis.grid(True, alpha=0.3)

    side_axis.set_xlim(0, container.length)
    side_axis.set_ylim(0, container.height)
    side_axis.set_xlabel("Length (units)")
    side_axis.set_ylabel("Height (units)")
    side_axis.set_title("Side view (XZ plane)")
    side_axis.set_aspect("equal")
    side_axis.grid(True, alpha=0.3)

    front_axis.set_xlim(0, container.width)
    front_axis.set_ylim(0, container.height)
    front_axis.set_xlabel("Width (units)")
    front_axis.set_ylabel("Height (units)")
    front_axis.set_title("Front view (YZ plane)")
    front_axis.set_aspect("equal")
    front_axis.grid(True, alpha=0.3)

    figure.tight_layout()
    plt.show()

## Reporting solution metrics and placements

The report separates packing outcomes from heuristic diagnostics. Utilization measures how much container volume the packed items occupy; the average position score summarizes the selected candidate locations and should not be interpreted as a direct measure of empty-space geometry.

In [ ]:
def print_blf_analysis(packed_items, container, placement_history):
    """Print packing metrics, heuristic diagnostics, and item placements."""
    metrics = calculate_solution_metrics(packed_items, container)

    print("\n" + "=" * 70)
    print("BOTTOM-LEFT-FILL 3D CONTAINER LOADING ANALYSIS")
    print("=" * 70)

    print(f"\nContainer: {container}")
    print(f"Container volume: {container.volume():.1f} cubic units")

    print("\nSolution results:")
    print(f"Items packed: {metrics['items_packed']}")
    print(f"Total packed volume: {metrics['total_volume']:.1f} cubic units")
    print(f"Total packed weight: {metrics['total_weight']:.1f} kg")
    print(f"Volume utilization: {metrics['volume_utilization']:.1f}%")
    if np.isfinite(container.max_weight):
        print(f"Weight utilization: {metrics['weight_utilization']:.1f}%")

    center = metrics["center_of_gravity"]
    print(
        "Center of gravity (length, width, height): "
        f"({center[0]:.2f}, {center[1]:.2f}, {center[2]:.2f}) units"
    )

    print("\nHeuristic diagnostics:")
    print(f"Successful placement decisions: {len(placement_history)}")
    if placement_history:
        average_support_points = np.mean(
            [entry["support_points_count"] for entry in placement_history]
        )
        average_position_score = np.mean(
            [entry["waste_score"] for entry in placement_history]
        )
        print(f"Average support points at placement: {average_support_points:.1f}")
        print(f"Average selected position score: {average_position_score:.4f}")

    print("\nDetailed item placements:")
    print("-" * 70)
    for item in packed_items:
        print(f"Item {item.item_id}:")
        print(f"  Priority: {item.calculate_priority():.3f}")
        print(f"  Position: {item.position}")
        print(f"  Orientation: {item.orientation}")
        print(f"  Volume: {item.volume():.1f} cubic units")
        print(f"  Weight: {item.weight} kg")
        print(f"  Density: {item.density():.3f} kg/cubic unit")
        print()

## Creating the example instance

The sum of item volumes is an upper bound on the packed volume, not by itself proof that all items can be loaded. A feasible arrangement is established by the algorithm's placement checks and can be inspected in the placement report and visualizations.

In [ ]:
container = Container(length=10, width=8, height=6, max_weight=1000)

items = [
    Item(4, 3, 2, 10, 1, 1.0),
    Item(3, 3, 3, 15, 2, 1.0),
    Item(2, 4, 3, 8, 3, 1.0),
]

print("3D CONTAINER LOADING - BOTTOM-LEFT-FILL HEURISTIC")
print("=" * 60)
print(f"Container: {container}")
print("\nItems to pack:")
for item in items:
    print(f"  {item}")

total_item_volume = sum(item.volume() for item in items)
total_item_weight = sum(item.weight for item in items)
volume_upper_bound = total_item_volume / container.volume() * 100

print(f"\nTotal item volume: {total_item_volume:.1f} cubic units")
print(f"Total item weight: {total_item_weight} kg")
print(f"Volume-utilization upper bound: {volume_upper_bound:.1f}%")

## Running BLF on the example

The random seed set near the beginning of the notebook makes the randomized ordering reproducible for a given execution environment. The returned `packed_items` list contains only successfully placed items; `placement_history` records the accepted decisions in sequence.

In [ ]:
blf_packer = BottomLeftFill3D(container, alpha=0.25)
packed_items = blf_packer.pack_items(items)
placement_history = blf_packer.placement_history

# These assertions verify that the reported example respects the model checks.
assert len({item.item_id for item in packed_items}) == len(packed_items)
assert sum(item.weight for item in packed_items) <= container.max_weight
for index, item in enumerate(packed_items):
    assert blf_packer.is_feasible_position(
        item, item.position, item.orientation
    )
    for other_item in packed_items[index + 1:]:
        assert not blf_packer.items_overlap(
            item, item.position, item.orientation, other_item
        )

## Interpreting the example results

A utilization percentage describes the volume of successfully placed items relative to the container volume. It does not indicate how balanced the load is or how much empty space is geometrically usable. The center-of-gravity coordinates and the detailed placement list provide complementary information, while the support rule remains only the simplified area-based check specified in the model.

In [ ]:
print_blf_analysis(packed_items, container, placement_history)

## Visual inspection

The projections help reveal overlaps or boundary violations that may be difficult to see in a single 3D view. The algorithm's own feasibility checks remain the formal checks used to accept placements.

In [ ]:
visualize_blf_solution(
    packed_items,
    container,
    "BLF heuristic - 3D container loading",
)

## Reading the algorithm trace

Each trace entry records the item priority, chosen coordinate and orientation, score, and number of support points available at the time the choice was made. The support percentage is recomputed against only items placed earlier in the sequence. A floor placement is fully supported by the floor.

In [ ]:
print("\n" + "=" * 70)
print("ALGORITHM TRACE ANALYSIS")
print("=" * 70)
print("\nStep-by-step placement decisions:")
print("-" * 50)

for step, decision in enumerate(placement_history, start=1):
    item_id = decision["item_id"]
    position = decision["position"]
    orientation = decision["orientation"]
    x, y, z = position
    length, width, _ = orientation

    print(f"\nStep {step}: Item {item_id}")
    print(f"  Priority score: {decision['priority']:.3f}")
    print(f"  Selected position: {position}")
    print(f"  Orientation: {orientation}")
    print(f"  Position score: {decision['waste_score']:.4f}")
    print(
        "  Support points available: "
        f"{decision['support_points_count']}"
    )

    if z == 0:
        print("  Support: container floor (100% of the bottom face)")
        continue

    support_area = 0
    for earlier_item in packed_items[:step - 1]:
        ex_x, ex_y, ex_z = earlier_item.position
        ex_length, ex_width, ex_height = earlier_item.orientation

        if abs(z - (ex_z + ex_height)) < 0.001:
            overlap_x = max(
                0,
                min(x + length, ex_x + ex_length) - max(x, ex_x),
            )
            overlap_y = max(
                0,
                min(y + width, ex_y + ex_width) - max(y, ex_y),
            )
            support_area += overlap_x * overlap_y

    support_percentage = support_area / (length * width) * 100
    print(f"  Support: {support_percentage:.1f}% of the bottom-face area")

## Sensitivity to `alpha` and a larger instance

The `alpha` experiment changes the amount of priority-order diversification while leaving the example data and heuristic unchanged. The same seed is reset for each tested value so that each run begins from the same pseudorandom sequence; the results are reproducible comparisons for this instance, not evidence that one value is best in general. In the larger-instance demonstration, the fixed seed also makes the generated item data reproducible.

In [ ]:
print("\n" + "=" * 70)
print("PERFORMANCE AND SENSITIVITY ANALYSIS")
print("=" * 70)

print("\nEffect of the randomness parameter (alpha):")
print("-" * 60)

alpha_values = [0.0, 0.1, 0.25, 0.5, 0.75]
results = []

for alpha in alpha_values:
    np.random.seed(42)
    blf_test = BottomLeftFill3D(container, alpha=alpha)
    test_packed = blf_test.pack_items(items)
    test_metrics = calculate_solution_metrics(test_packed, container)

    result = {
        "alpha": alpha,
        "utilization": test_metrics["volume_utilization"],
        "items_packed": test_metrics["items_packed"],
    }
    results.append(result)

    print(
        f"Alpha = {alpha:.2f}: "
        f"{test_metrics['volume_utilization']:.1f}% utilization, "
        f"{test_metrics['items_packed']} items packed"
    )

best_result = max(results, key=lambda result: result["utilization"])
tied_results = [
    result
    for result in results
    if np.isclose(
        result["utilization"],
        best_result["utilization"],
        rtol=1e-9,
        atol=1e-9,
    )
]
tied_alphas = [result["alpha"] for result in tied_results]
print(
    "\nHighest observed utilization: "
    f"{best_result['utilization']:.1f}% at alpha value(s) {tied_alphas}"
)

print("\nScalability demonstration with a larger instance:")
print("-" * 50)

np.random.seed(42)
large_items = []
for item_index in range(10):
    length = np.random.randint(1, 5)
    width = np.random.randint(1, 4)
    height = np.random.randint(1, 4)
    weight = np.random.randint(5, 25)
    urgency = np.random.uniform(0.8, 1.2)
    large_items.append(
        Item(length, width, height, weight, item_index + 1, urgency)
    )

print(f"Generated items: {len(large_items)}")
print(
    "Total item volume: "
    f"{sum(item.volume() for item in large_items):.1f} cubic units"
)

start_time = time.time()
blf_large = BottomLeftFill3D(container, alpha=0.25)
large_packed = blf_large.pack_items(large_items)
elapsed_time = time.time() - start_time

large_metrics = calculate_solution_metrics(large_packed, container)
print("\nLarger-instance results:")
print(f"Items packed: {large_metrics['items_packed']}/{len(large_items)}")
print(f"Volume utilization: {large_metrics['volume_utilization']:.1f}%")
print(f"Measured computation time: {elapsed_time:.3f} seconds")
print(
    "Measured average time per input item: "
    f"{elapsed_time / len(large_items):.3f} seconds"
)

## Assessing the quality gap for the three-item instance

For a fixed set of items, the sum of all item volumes is an upper bound on packed volume: no feasible plan can contain more volume than the total volume of the available items. If a feasible plan packs every item, it reaches that upper bound and is therefore optimal for the specific objective of maximizing packed volume on this instance. This argument does not establish optimality for other objectives, such as minimizing height, balancing the load, or preserving unloading access.

The code below checks whether all items were packed before using the volume upper bound as an optimal value. If an item is left unpacked, the upper bound alone does not prove that the heuristic has found an optimum, so no optimality gap is reported.

In [ ]:
print("\n" + "=" * 70)
print("VOLUME-OBJECTIVE QUALITY-GAP ANALYSIS")
print("=" * 70)

blf_metrics = calculate_solution_metrics(packed_items, container)
all_items_packed = blf_metrics["items_packed"] == len(items)

print("\nThree-item instance:")
print(f"BLF volume utilization: {blf_metrics['volume_utilization']:.1f}%")
print(f"Items packed: {blf_metrics['items_packed']}/{len(items)}")

if all_items_packed:
    optimal_utilization = sum(item.volume() for item in items)
    optimal_utilization = optimal_utilization / container.volume() * 100
    absolute_gap = max(
        0.0, optimal_utilization - blf_metrics["volume_utilization"]
    )
    relative_gap = (
        absolute_gap / optimal_utilization * 100
        if not np.isclose(optimal_utilization, 0.0)
        else 0.0
    )

    print(
        "Volume upper bound, attained by packing every item: "
        f"{optimal_utilization:.1f}%"
    )
    print(f"Absolute gap: {absolute_gap:.1f} percentage points")
    print(f"Relative gap: {relative_gap:.1f}%")
    print("Conclusion: the plan is volume-optimal for this instance.")
else:
    volume_upper_bound = sum(item.volume() for item in items)
    volume_upper_bound = volume_upper_bound / container.volume() * 100
    print(f"Available-item volume upper bound: {volume_upper_bound:.1f}%")
    print(
        "The upper bound is not attained, so this comparison does not "
        "establish an optimality gap."
    )

## Why use this heuristic tier?

The BLF method makes explicit, sequential placement decisions using a priority order, a finite set of support points, and a feasibility check. It is useful when a quickly constructed feasible plan is valuable and a full search over arrangements is impractical. Its result depends on the candidate-point generator, pruning rule, item order, and position score; it has no general guarantee of optimality or of a fixed percentage of the optimum.

### Main design choices in this implementation

- **Priority rule:** volume × density × urgency factor orders the items before placement.
- **Candidate generation:** corners and selected edge midpoints of placed items provide possible lower-corner coordinates.
- **Placement evaluation:** all available candidate points and distinct axis-aligned orientations are checked against boundaries, overlap, weight capacity, and the support-area threshold.
- **Position ranking:** a weighted score favors low positions and positions near the container origin.
- **Candidate-list control:** when the support-point list grows beyond the threshold, it is shortened using height and horizontal-distance ranking. This saves search effort but may discard useful candidates.
- **Diversification:** `alpha` controls randomized adjacent swaps in priority order. It can produce different feasible plans, but does not itself provide an optimality guarantee.

### Practical interpretation and limitations

The calculation is polynomial in the number of items and the number of retained candidate points for this implementation, but that does not mean every large instance will be solved quickly or well: the candidate count, geometric checks, and chosen pruning rule affect practical performance. The example's measured runtime is environment-dependent. Likewise, results from one small instance or a small `alpha` experiment should not be generalized into a universal quality guarantee.

The support-area rule is a geometric proxy rather than a full mechanical stability model. The notebook does not model item fragility, stacking-strength limits, load distribution within an item, container-wall strength, vehicle dynamics, or unloading sequence. These can be introduced as additional constraints when the application requires them, while retaining the same high-level priority-based, support-point BLF workflow.